# US Job Market Visualizer — Karpathy BLS Project

A recreation of [karpathy/jobs](https://github.com/karpathy/jobs) in Google Colab.

This notebook:
1. **Clones** the repo (which already has all 342 scraped BLS pages, parsed CSV, and LLM scores)
2. **Visualizes** the data as interactive treemaps (4 coloring modes)
3. **Exposes the full pipeline** so you can re-run any step:
   - Scrape BLS Occupational Outlook Handbook pages (Playwright)
   - Parse HTML → Markdown
   - Build `occupations.csv`
   - Score AI exposure via OpenRouter LLM
   - Generate `prompt.md` for direct LLM analysis

**Original project:** https://github.com/karpathy/jobs  
**Live visualization:** https://karpathy.ai/jobs/

## 1. Setup

In [ ]:
# Install required packages
!pip install -q beautifulsoup4 httpx plotly python-dotenv

In [ ]:
import os

REPO_DIR = "/content/jobs"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/karpathy/jobs {REPO_DIR}
else:
    print("Repo already cloned.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir("."))

## 2. Load & Explore Data

The repo already contains all pre-scraped and pre-scored data:
- `occupations.csv` — 342 occupations with BLS statistics
- `scores.json` — AI exposure scores (0–10) from Gemini Flash
- `html/` — raw scraped HTML from BLS website (342 files)
- `occupations.json` — master occupation list with URLs

In [ ]:
import csv
import json
import pandas as pd

# Load occupations CSV
df = pd.read_csv("occupations.csv")

# Load AI exposure scores
with open("scores.json") as f:
    scores_list = json.load(f)
scores = {s["slug"]: s for s in scores_list}

# Merge scores into DataFrame
df["exposure"] = df["slug"].map(lambda s: scores.get(s, {}).get("exposure"))
df["exposure_rationale"] = df["slug"].map(lambda s: scores.get(s, {}).get("rationale", ""))

print(f"Loaded {len(df)} occupations")
print(f"\nColumns: {list(df.columns)}")
df.head(3)

In [ ]:
# Quick stats
total_jobs = df["num_jobs_2024"].sum()
avg_pay = df["median_pay_annual"].mean()
avg_exposure = df["exposure"].mean()

print("=" * 50)
print(f"Total occupations:     {len(df)}")
print(f"Total jobs:            {total_jobs:,.0f} ({total_jobs/1e6:.0f}M)")
print(f"Avg median pay:        ${avg_pay:,.0f}/year")
print(f"Avg AI exposure:       {avg_exposure:.2f}/10")
print("=" * 50)

# AI exposure distribution
print("\nAI Exposure distribution:")
exposure_counts = df["exposure"].value_counts().sort_index()
for score, count in exposure_counts.items():
    bar = "█" * count
    print(f"  {int(score):2d}/10: {bar} ({count})")

In [ ]:
# Tier breakdown (matches prompt.md analysis)
tiers = [
    ("Minimal (0–1)",    0,  1),
    ("Low (2–3)",        2,  3),
    ("Moderate (4–5)",   4,  5),
    ("High (6–7)",       6,  7),
    ("Very high (8–10)", 8, 10),
]

total_wages = (df["num_jobs_2024"] * df["median_pay_annual"]).sum()

print(f"{'Tier':<22} {'Occs':>5} {'Jobs':>10} {'% Jobs':>8} {'Avg Pay':>10}")
print("-" * 60)
for name, lo, hi in tiers:
    g = df[(df["exposure"] >= lo) & (df["exposure"] <= hi)]
    jobs = g["num_jobs_2024"].sum()
    wages = (g["num_jobs_2024"] * g["median_pay_annual"]).sum()
    avg_pay_g = wages / jobs if jobs > 0 else 0
    print(f"{name:<22} {len(g):>5} {jobs:>10,.0f} {jobs/total_jobs*100:>7.1f}% ${avg_pay_g:>9,.0f}")

## 3. Interactive Treemap Visualizations

Treemaps where **area = number of jobs**, colored by four different metrics.

Matches the original [karpathy.ai/jobs](https://karpathy.ai/jobs/) interface.

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

# Build a clean merged DataFrame for plotting
plot_df = df.copy()
plot_df = plot_df.dropna(subset=["num_jobs_2024"])
plot_df["num_jobs_2024"] = plot_df["num_jobs_2024"].astype(int)

# Pretty-format category labels
plot_df["category_label"] = plot_df["category"].str.replace("-", " ").str.title()

# Add root node (required by Plotly treemap)
plot_df["root"] = "All Occupations"

# Education shorthand mapping
edu_map = {
    "No formal educational credential": "No formal",
    "High school diploma or equivalent": "HS diploma",
    "Some college, no degree": "Some college",
    "Associate's degree": "Associate's",
    "Postsecondary nondegree award": "Postsecondary",
    "Bachelor's degree": "Bachelor's",
    "Master's degree": "Master's",
    "Doctoral or professional degree": "Doctoral/Prof",
    "See How to Become One": "Varies",
}
edu_order = ["No formal", "HS diploma", "Some college", "Postsecondary",
             "Associate's", "Bachelor's", "Master's", "Doctoral/Prof", "Varies"]

plot_df["edu_short"] = plot_df["entry_education"].map(edu_map).fillna(plot_df["entry_education"])
plot_df["edu_rank"] = plot_df["edu_short"].map({e: i for i, e in enumerate(edu_order)}).fillna(5)

print(f"Plotting {len(plot_df)} occupations across {plot_df['category_label'].nunique()} categories")

In [ ]:
# ── Treemap 1: AI Exposure ──

mask = plot_df["exposure"].notna()
d = plot_df[mask].copy()

# Build hover text
d["hover"] = (
    "<b>" + d["title"] + "</b><br>"
    + "AI Exposure: " + d["exposure"].astype(int).astype(str) + "/10<br>"
    + "Jobs: " + d["num_jobs_2024"].apply(lambda x: f"{x:,}") + "<br>"
    + "Pay: $" + d["median_pay_annual"].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "?") + "/yr<br>"
    + "<i>" + d["exposure_rationale"].str[:120] + "...</i>"
)

fig1 = go.Figure(go.Treemap(
    labels=list(d["title"]) + list(d["category_label"].unique()) + ["All Occupations"],
    parents=list(d["category_label"]) + ["All Occupations"] * d["category_label"].nunique() + [""],
    values=list(d["num_jobs_2024"]) + [0] * d["category_label"].nunique() + [0],
    customdata=list(d["hover"]) + [""] * d["category_label"].nunique() + [""],
    hovertemplate="%{customdata}<extra></extra>",
    marker=dict(
        colors=list(d["exposure"]) + [None] * d["category_label"].nunique() + [None],
        colorscale=[
            [0.0,  "#1a9641"],   # 0 = green (low exposure)
            [0.5,  "#ffffbf"],   # 5 = yellow
            [1.0,  "#d7191c"],   # 10 = red (high exposure)
        ],
        cmin=0, cmax=10,
        colorbar=dict(title="AI Exposure", tickvals=list(range(0, 11))),
        showscale=True,
    ),
    textinfo="label",
    pathbar_visible=True,
))

fig1.update_layout(
    title="US Job Market — AI Exposure Score (0=low, 10=high)<br><sup>Area = number of jobs · Color = AI exposure</sup>",
    height=700,
    margin=dict(t=80, l=10, r=10, b=10),
)
fig1.show()

In [ ]:
# ── Treemap 2: BLS Growth Projection ──

mask = plot_df["outlook_pct"].notna()
d = plot_df[mask].copy()

d["hover"] = (
    "<b>" + d["title"] + "</b><br>"
    + "Growth: " + d["outlook_pct"].apply(lambda x: f"{x:+.0f}%") + " (" + d["outlook_desc"] + ")<br>"
    + "Jobs (2024): " + d["num_jobs_2024"].apply(lambda x: f"{x:,}") + "<br>"
    + "Pay: $" + d["median_pay_annual"].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "?") + "/yr"
)

fig2 = go.Figure(go.Treemap(
    labels=list(d["title"]) + list(d["category_label"].unique()) + ["All Occupations"],
    parents=list(d["category_label"]) + ["All Occupations"] * d["category_label"].nunique() + [""],
    values=list(d["num_jobs_2024"]) + [0] * d["category_label"].nunique() + [0],
    customdata=list(d["hover"]) + [""] * d["category_label"].nunique() + [""],
    hovertemplate="%{customdata}<extra></extra>",
    marker=dict(
        colors=list(d["outlook_pct"]) + [None] * d["category_label"].nunique() + [None],
        colorscale=[
            [0.0, "#d7191c"],   # red = decline
            [0.4, "#ffffbf"],   # yellow = flat
            [1.0, "#1a9641"],   # green = growth
        ],
        cmin=-20, cmax=30,
        colorbar=dict(title="Growth %"),
        showscale=True,
    ),
    textinfo="label",
    pathbar_visible=True,
))

fig2.update_layout(
    title="US Job Market — BLS Projected Job Growth (2024–2034)<br><sup>Area = number of jobs · Color = projected % change</sup>",
    height=700,
    margin=dict(t=80, l=10, r=10, b=10),
)
fig2.show()

In [ ]:
# ── Treemap 3: Median Annual Pay ──

mask = plot_df["median_pay_annual"].notna()
d = plot_df[mask].copy()

d["hover"] = (
    "<b>" + d["title"] + "</b><br>"
    + "Median pay: $" + d["median_pay_annual"].apply(lambda x: f"{x:,.0f}") + "/yr<br>"
    + "Jobs: " + d["num_jobs_2024"].apply(lambda x: f"{x:,}") + "<br>"
    + "Outlook: " + d["outlook_pct"].apply(lambda x: f"{x:+.0f}%" if pd.notna(x) else "?")
)

fig3 = go.Figure(go.Treemap(
    labels=list(d["title"]) + list(d["category_label"].unique()) + ["All Occupations"],
    parents=list(d["category_label"]) + ["All Occupations"] * d["category_label"].nunique() + [""],
    values=list(d["num_jobs_2024"]) + [0] * d["category_label"].nunique() + [0],
    customdata=list(d["hover"]) + [""] * d["category_label"].nunique() + [""],
    hovertemplate="%{customdata}<extra></extra>",
    marker=dict(
        colors=list(d["median_pay_annual"]) + [None] * d["category_label"].nunique() + [None],
        colorscale="Viridis",
        cmin=20000, cmax=220000,
        colorbar=dict(
            title="Annual Pay",
            tickvals=[20000, 50000, 100000, 150000, 200000],
            ticktext=["$20K", "$50K", "$100K", "$150K", "$200K"],
        ),
        showscale=True,
    ),
    textinfo="label",
    pathbar_visible=True,
))

fig3.update_layout(
    title="US Job Market — Median Annual Salary<br><sup>Area = number of jobs · Color = median annual pay</sup>",
    height=700,
    margin=dict(t=80, l=10, r=10, b=10),
)
fig3.show()

In [ ]:
# ── Treemap 4: Education Level ──

d = plot_df.copy()

edu_colors = {
    "No formal":     "#9e0142",
    "HS diploma":    "#d53e4f",
    "Some college":  "#f46d43",
    "Postsecondary": "#fdae61",
    "Associate's":   "#fee08b",
    "Bachelor's":    "#66c2a5",
    "Master's":      "#3288bd",
    "Doctoral/Prof": "#5e4fa2",
    "Varies":        "#aaaaaa",
}

d["edu_color"] = d["edu_short"].map(edu_colors).fillna("#cccccc")

d["hover"] = (
    "<b>" + d["title"] + "</b><br>"
    + "Education: " + d["entry_education"].fillna("?") + "<br>"
    + "Jobs: " + d["num_jobs_2024"].apply(lambda x: f"{x:,}") + "<br>"
    + "Pay: $" + d["median_pay_annual"].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "?") + "/yr"
)

all_cats = list(d["category_label"].unique())

fig4 = go.Figure(go.Treemap(
    labels=list(d["title"]) + all_cats + ["All Occupations"],
    parents=list(d["category_label"]) + ["All Occupations"] * len(all_cats) + [""],
    values=list(d["num_jobs_2024"]) + [0] * len(all_cats) + [0],
    customdata=list(d["hover"]) + [""] * len(all_cats) + [""],
    hovertemplate="%{customdata}<extra></extra>",
    marker=dict(
        colors=list(d["edu_color"]) + ["#eeeeee"] * len(all_cats) + ["#eeeeee"],
    ),
    textinfo="label",
    pathbar_visible=True,
))

# Legend as annotation
legend_text = "  ".join([f"<span style='color:{c}'>■</span> {e}" for e, c in edu_colors.items()])

fig4.update_layout(
    title="US Job Market — Education Requirement<br><sup>Area = number of jobs · Color = entry-level education required</sup>",
    height=700,
    margin=dict(t=80, l=10, r=10, b=60),
    annotations=[
        dict(
            x=0, y=-0.08, xref="paper", yref="paper",
            text=" | ".join([f"<span style='color:{c}'>■ {e}</span>" for e, c in list(edu_colors.items())[:5]]),
            showarrow=False, font=dict(size=11), align="left"
        )
    ]
)
fig4.show()

# Print legend
print("\nEducation color legend:")
for edu, color in edu_colors.items():
    count = (d["edu_short"] == edu).sum()
    jobs = d[d["edu_short"] == edu]["num_jobs_2024"].sum()
    print(f"  {color}  {edu:<16} {count:>3} occupations  {jobs:>10,} jobs")

## 4. Re-running the Pipeline (Optional)

The cells below let you re-run each step from scratch.

### Pipeline Overview

```
occupational_outlook_handbook.html          ← BLS A-Z index page (already in repo)
         │
         ▼
  parse_occupations.py  →  occupations.json (342 occupation URLs)
         │
         ▼
    scrape.py           →  html/<slug>.html  (raw BLS pages, already in repo)
         │
         ▼
    process.py          →  pages/<slug>.md   (clean Markdown)
         │
         ▼
    make_csv.py         →  occupations.csv   (structured stats)
         │
         ▼
    score.py            →  scores.json       (LLM AI-exposure scores)
         │
         ▼
 build_site_data.py     →  site/data.json    (merged site data)
         │
         ▼
   make_prompt.py       →  prompt.md         (LLM analysis document)
```

### Step 1: Parse BLS A-Z Index → `occupations.json`

Extracts all 342 occupations from the BLS Occupational Outlook Handbook index page.

In [ ]:
# This uses the already-downloaded occupational_outlook_handbook.html
# To re-download: !curl -o occupational_outlook_handbook.html https://www.bls.gov/ooh/a-z-index.htm

from bs4 import BeautifulSoup
import json, re

with open("occupational_outlook_handbook.html") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

az_list = soup.find("div", class_="a-z-list")
occupations = {}  # url -> title

for li in az_list.find_all("li"):
    links = li.find_all("a")
    text = li.get_text()
    if ", see:" in text or ", see " in text:
        if len(links) >= 2:
            url = links[1]["href"]
            name = links[1].get_text(strip=True)
            if url not in occupations:
                occupations[url] = name
    else:
        if links:
            url = links[0]["href"]
            name = links[0].get_text(strip=True)
            if url not in occupations:
                occupations[url] = name

sorted_occs = sorted(occupations.items(), key=lambda x: x[1].lower())

# Extract slug from URL (e.g. /ooh/management/accountants/ -> accountants)
def url_to_slug(url):
    parts = [p for p in url.split("/") if p and p not in ("ooh", "")]
    return parts[-1] if parts else url

output = []
for url, name in sorted_occs:
    slug = url_to_slug(url)
    category = url.split("/")[2] if len(url.split("/")) > 3 else ""
    output.append({"title": name, "slug": slug, "category": category, "url": f"https://www.bls.gov{url}"})

print(f"Parsed {len(output)} unique occupations")
print("First 5:", [o["title"] for o in output[:5]])

### Step 2: Scrape BLS Pages → `html/<slug>.html`

Uses Playwright to scrape BLS pages. The repo already has all 342 cached — only run if you want to refresh.

In [ ]:
# Install Playwright and Chromium
# Only needed if you want to re-scrape
!pip install -q playwright
!playwright install chromium
!playwright install-deps chromium

In [ ]:
# Scrape a small subset to test (set FORCE=True to re-scrape cached files)
# The full run of 342 pages takes ~10-15 minutes with a 1s delay

import time, os
from playwright.sync_api import sync_playwright

def scrape_occupations(occupations_list, start=0, end=5, force=False, delay=1.0):
    """Scrape BLS OOH pages, save raw HTML to html/<slug>.html"""
    os.makedirs("html", exist_ok=True)
    subset = occupations_list[start:end]

    to_scrape = []
    for i, occ in enumerate(subset, start=start):
        html_path = f"html/{occ['slug']}.html"
        if not force and os.path.exists(html_path):
            print(f"  [{i}] CACHED {occ['title']}")
        else:
            to_scrape.append((i, occ))

    if not to_scrape:
        print("Nothing to scrape — all cached.")
        return

    print(f"Scraping {len(to_scrape)} pages...")

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)  # headless=True for Colab
        page = browser.new_page()

        for idx, (i, occ) in enumerate(to_scrape):
            slug, url = occ["slug"], occ["url"]
            html_path = f"html/{slug}.html"
            print(f"  [{i}] {occ['title']}...", end=" ", flush=True)
            try:
                resp = page.goto(url, wait_until="domcontentloaded", timeout=15000)
                if resp.status != 200:
                    print(f"HTTP {resp.status} — SKIPPED")
                    continue
                with open(html_path, "w") as f:
                    f.write(page.content())
                print("OK")
            except Exception as e:
                print(f"ERROR: {e}")
            if idx < len(to_scrape) - 1:
                time.sleep(delay)

        browser.close()

# Load current occupations list
with open("occupations.json") as f:
    occupations_list = json.load(f)

# Scrape first 3 as a test (set force=True to actually re-fetch)
# scrape_occupations(occupations_list, start=0, end=3, force=False)
print(f"Repo already has {len(os.listdir('html'))} cached HTML files.")
print("Uncomment the scrape_occupations() call above to re-scrape.")

### Step 3: Process HTML → Markdown (`pages/<slug>.md`)

In [ ]:
import re
from bs4 import BeautifulSoup

def clean(text):
    return re.sub(r'\s+', ' ', text).strip()

def parse_ooh_page(html_path):
    """Parse a BLS OOH detail page HTML into clean Markdown."""
    with open(html_path) as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    md = []

    h1 = soup.find("h1")
    title = clean(h1.get_text()) if h1 else "Unknown"
    md.append(f"# {title}\n")

    canonical = soup.find("link", rel="canonical")
    if canonical:
        md.append(f"**Source:** {canonical['href']}\n")

    # Quick Facts table
    qf = soup.find("table", id="quickfacts")
    if qf:
        md.append("## Quick Facts\n")
        md.append("| Field | Value |")
        md.append("|-------|-------|")
        for row in qf.find("tbody").find_all("tr"):
            th, td = row.find("th"), row.find("td")
            if th and td:
                md.append(f"| {clean(th.get_text())} | {clean(td.get_text())} |")
        md.append("")

    # Tab sections
    panes = soup.find("div", id="panes")
    if panes:
        for tab_id in ["tab-2", "tab-3", "tab-4", "tab-5", "tab-6"]:
            tab_div = panes.find("div", id=tab_id)
            if not tab_div:
                continue
            article = tab_div.find("article") or tab_div
            h2 = article.find("h2")
            if not h2:
                continue
            section = clean(h2.find("span").get_text()) if h2.find("span") else clean(h2.get_text())
            md.append(f"## {section}\n")
            for elem in article.children:
                if not hasattr(elem, 'name'):
                    continue
                if elem.name == 'h3':
                    md.append(f"### {clean(elem.get_text())}\n")
                elif elem.name == 'p':
                    t = clean(elem.get_text())
                    if t:
                        md.append(f"{t}\n")
                elif elem.name == 'ul':
                    for li in elem.find_all("li"):
                        md.append(f"- {clean(li.get_text())}")
                    md.append("")

    return "\n".join(md)


def process_all(force=False):
    """Convert all html/<slug>.html to pages/<slug>.md"""
    os.makedirs("pages", exist_ok=True)
    with open("occupations.json") as f:
        occs = json.load(f)

    processed = skipped = missing = 0
    for occ in occs:
        slug = occ["slug"]
        html_path = f"html/{slug}.html"
        md_path = f"pages/{slug}.md"
        if not os.path.exists(html_path):
            missing += 1
            continue
        if not force and os.path.exists(md_path):
            skipped += 1
            continue
        with open(md_path, "w") as f:
            f.write(parse_ooh_page(html_path))
        processed += 1

    print(f"Processed: {processed}  Skipped (cached): {skipped}  Missing HTML: {missing}")


# Preview: parse one page
sample_html = "html/software-developers.html"
if os.path.exists(sample_html):
    sample_md = parse_ooh_page(sample_html)
    print(sample_md[:1500])
    print("...")

# Uncomment to process all HTML files
# process_all(force=False)

### Step 4: Build `occupations.csv`

In [ ]:
import csv
from bs4 import BeautifulSoup

def parse_pay(value):
    annual = hourly = ""
    amounts = re.findall(r'\$([\d,]+(?:\.\d+)?)', value)
    if "per year" in value and "per hour" in value and len(amounts) >= 2:
        annual = amounts[0].replace(",", "")
        hourly = amounts[1].replace(",", "")
    elif "per year" in value and amounts:
        annual = amounts[0].replace(",", "")
    elif "per hour" in value and amounts:
        hourly = amounts[0].replace(",", "")
    return annual, hourly

def parse_outlook(value):
    m = re.match(r'(-?\d+)%\s*\((.+)\)', value)
    if m:
        return m.group(1), m.group(2)
    m = re.match(r'(-?\d+)%', value)
    return (m.group(1), "") if m else ("", value.strip())

def extract_row(html_path, occ_meta):
    with open(html_path) as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    row = {k: "" for k in ["title","category","slug","url","soc_code",
                             "median_pay_annual","median_pay_hourly","entry_education",
                             "work_experience","training","num_jobs_2024",
                             "projected_employment_2034","outlook_pct","outlook_desc","employment_change"]}
    row.update({"title": occ_meta["title"], "category": occ_meta.get("category",""),
                "slug": occ_meta["slug"], "url": occ_meta["url"]})

    qf = soup.find("table", id="quickfacts")
    if qf:
        for tr in qf.find("tbody").find_all("tr"):
            th, td = tr.find("th"), tr.find("td")
            if not th or not td:
                continue
            field = clean(th.get_text()).lower()
            value = clean(td.get_text())
            if "median pay" in field:
                row["median_pay_annual"], row["median_pay_hourly"] = parse_pay(value)
            elif "entry-level education" in field:
                row["entry_education"] = value
            elif "work experience" in field:
                row["work_experience"] = value
            elif "on-the-job training" in field:
                row["training"] = value
            elif "number of jobs" in field:
                row["num_jobs_2024"] = value.replace(",", "")
            elif "job outlook" in field:
                row["outlook_pct"], row["outlook_desc"] = parse_outlook(value)
            elif "employment change" in field:
                row["employment_change"] = value.replace(",", "")

    ot = soup.find("table", id="outlook-table")
    if ot:
        tbody = ot.find("tbody")
        if tbody:
            tr = tbody.find("tr")
            if tr:
                cells = [clean(c.get_text()) for c in tr.find_all(["td","th"])]
                if len(cells) >= 4:
                    if cells[1] != "—":
                        row["soc_code"] = cells[1]
                    row["projected_employment_2034"] = cells[3].replace(",", "")

    return row


def build_csv(output_path="occupations.csv"):
    """Build CSV from all html/ files."""
    with open("occupations.json") as f:
        occs = json.load(f)

    fieldnames = ["title","category","slug","soc_code","median_pay_annual",
                  "median_pay_hourly","entry_education","work_experience","training",
                  "num_jobs_2024","projected_employment_2034","outlook_pct",
                  "outlook_desc","employment_change","url"]
    rows = []
    for occ in occs:
        hp = f"html/{occ['slug']}.html"
        if os.path.exists(hp):
            rows.append(extract_row(hp, occ))

    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Wrote {len(rows)} rows to {output_path}")
    return rows


# Uncomment to rebuild CSV
# rows = build_csv()
print("occupations.csv already exists with", len(pd.read_csv('occupations.csv')), "rows")

### Step 5: Score AI Exposure via OpenRouter LLM → `scores.json`

Requires an [OpenRouter API key](https://openrouter.ai/). The repo already has scores from `google/gemini-3-flash-preview`.

In [ ]:
import httpx, getpass, time, json

SCORE_SYSTEM_PROMPT = """\
You are an expert analyst evaluating how exposed different occupations are to AI.
You will be given a detailed description of an occupation from the Bureau of Labor Statistics.

Rate the occupation's overall **AI Exposure** on a scale from 0 to 10.

AI Exposure measures: how much will AI reshape this occupation? Consider both direct effects
(AI automating tasks currently done by humans) and indirect effects (AI making each worker
so productive that fewer are needed).

A key signal is whether the job's work product is fundamentally digital. If the job can be
done entirely from a home office on a computer — writing, coding, analyzing, communicating —
then AI exposure is inherently high (7+). Conversely, jobs requiring physical presence,
manual skill, or real-time human interaction have a natural barrier.

Calibration anchors:
- 0-1 Minimal: roofer, landscaper (almost entirely physical)
- 2-3 Low: electrician, plumber, firefighter
- 4-5 Moderate: nurse, police officer, veterinarian
- 6-7 High: teacher, accountant, journalist
- 8-9 Very high: software developer, graphic designer, paralegal
- 10 Maximum: data entry clerk, telemarketer

Respond with ONLY valid JSON, no other text:
{"exposure": <0-10>, "rationale": "<2-3 sentences>"}\
"""

def score_one(client, text, api_key, model="google/gemini-flash-1.5"):
    response = client.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": SCORE_SYSTEM_PROMPT},
                {"role": "user", "content": text},
            ],
            "temperature": 0.2,
        },
        timeout=60,
    )
    response.raise_for_status()
    content = response.json()["choices"][0]["message"]["content"].strip()
    if content.startswith("```"):
        content = content.split("\n", 1)[1]
        if content.endswith("```"):
            content = content[:-3]
    return json.loads(content.strip())


def score_occupations(start=0, end=5, force=False, delay=0.5, model="google/gemini-flash-1.5"):
    """Score occupations with an LLM via OpenRouter."""
    api_key = getpass.getpass("Enter your OpenRouter API key: ")

    with open("occupations.json") as f:
        occs = json.load(f)

    # Load existing scores
    scores = {}
    if os.path.exists("scores.json") and not force:
        with open("scores.json") as f:
            for entry in json.load(f):
                scores[entry["slug"]] = entry

    subset = occs[start:end]
    client = httpx.Client()

    for i, occ in enumerate(subset):
        slug = occ["slug"]
        if slug in scores:
            print(f"  [{i+1}] CACHED {occ['title']} (exposure={scores[slug]['exposure']})")
            continue

        md_path = f"pages/{slug}.md"
        if not os.path.exists(md_path):
            print(f"  [{i+1}] SKIP {slug} (no markdown — run Step 3 first)")
            continue

        with open(md_path) as f:
            text = f.read()

        print(f"  [{i+1}/{len(subset)}] {occ['title']}...", end=" ", flush=True)
        try:
            result = score_one(client, text, api_key, model)
            scores[slug] = {"slug": slug, "title": occ["title"], **result}
            print(f"exposure={result['exposure']}")
        except Exception as e:
            print(f"ERROR: {e}")

        # Save incrementally
        with open("scores.json", "w") as f:
            json.dump(list(scores.values()), f, indent=2)

        if i < len(subset) - 1:
            time.sleep(delay)

    client.close()
    print(f"Done. {len(scores)} total scores in scores.json")


# To score occupations 0-4 (test run):
# score_occupations(start=0, end=5)
#
# To score all 342:
# score_occupations(start=0, end=342)

print(f"scores.json already has {len(scores_list)} scored occupations.")
print("Uncomment the score_occupations() call above to re-score.")

### Step 6: Build `site/data.json` (merged data for the website)

In [ ]:
def build_site_data():
    with open("scores.json") as f:
        scores_map = {s["slug"]: s for s in json.load(f)}

    rows = list(csv.DictReader(open("occupations.csv")))

    data = []
    for row in rows:
        slug = row["slug"]
        score = scores_map.get(slug, {})
        data.append({
            "title": row["title"],
            "slug": slug,
            "category": row["category"],
            "pay": int(row["median_pay_annual"]) if row["median_pay_annual"] else None,
            "jobs": int(row["num_jobs_2024"]) if row["num_jobs_2024"] else None,
            "outlook": int(row["outlook_pct"]) if row["outlook_pct"] else None,
            "outlook_desc": row["outlook_desc"],
            "education": row["entry_education"],
            "exposure": score.get("exposure"),
            "exposure_rationale": score.get("rationale"),
            "url": row.get("url", ""),
        })

    os.makedirs("site", exist_ok=True)
    with open("site/data.json", "w") as f:
        json.dump(data, f)

    total_jobs = sum(d["jobs"] for d in data if d["jobs"])
    print(f"Wrote {len(data)} occupations to site/data.json")
    print(f"Total jobs represented: {total_jobs:,}")
    return data


site_data = build_site_data()

## 5. Generate `prompt.md` — Consolidated LLM Analysis Document

A single Markdown file containing all project data designed for direct LLM analysis.

In [ ]:
def fmt_pay(pay):
    return f"${pay:,}" if pay else "?"

def fmt_jobs(jobs):
    if jobs is None:
        return "?"
    if jobs >= 1_000_000:
        return f"{jobs/1e6:.1f}M"
    if jobs >= 1_000:
        return f"{jobs/1e3:.0f}K"
    return str(jobs)

def generate_prompt_md():
    with open("occupations.json") as f:
        occs_list = json.load(f)

    csv_rows = {row["slug"]: row for row in csv.DictReader(open("occupations.csv"))}
    scores_map = {s["slug"]: s for s in json.load(open("scores.json"))}

    records = []
    for occ in occs_list:
        slug = occ["slug"]
        row = csv_rows.get(slug, {})
        score = scores_map.get(slug, {})
        records.append({
            "title": occ["title"],
            "slug": slug,
            "category": row.get("category", ""),
            "pay": int(row["median_pay_annual"]) if row.get("median_pay_annual") else None,
            "jobs": int(row["num_jobs_2024"]) if row.get("num_jobs_2024") else None,
            "outlook_pct": int(row["outlook_pct"]) if row.get("outlook_pct") else None,
            "outlook_desc": row.get("outlook_desc", ""),
            "education": row.get("entry_education", ""),
            "exposure": score.get("exposure"),
            "rationale": score.get("rationale", ""),
        })

    records.sort(key=lambda r: (-(r["exposure"] or 0), -(r["jobs"] or 0)))

    lines = [
        "# AI Exposure of the US Job Market",
        "",
        "Structured data on 342 US occupations from the BLS Occupational Outlook Handbook,",
        "each scored for AI exposure on a 0-10 scale by an LLM (Gemini Flash).",
        "",
        f"GitHub: https://github.com/karpathy/jobs",
        f"Live visualization: https://karpathy.ai/jobs/",
        "",
    ]

    # Aggregate stats
    total_jobs = sum(r["jobs"] or 0 for r in records)
    total_wages = sum((r["jobs"] or 0) * (r["pay"] or 0) for r in records)
    w_sum = sum((r["exposure"] or 0) * (r["jobs"] or 0) for r in records if r["exposure"] and r["jobs"])
    w_cnt = sum(r["jobs"] or 0 for r in records if r["exposure"] and r["jobs"])
    w_avg = w_sum / w_cnt if w_cnt else 0

    lines += [
        "## Aggregate Statistics",
        "",
        f"- Total occupations: {len(records)}",
        f"- Total jobs: {total_jobs:,} ({total_jobs/1e6:.0f}M)",
        f"- Total annual wages: ${total_wages/1e12:.1f}T",
        f"- Job-weighted average AI exposure: {w_avg:.1f}/10",
        "",
    ]

    # All occupations table grouped by exposure
    lines += ["## All 342 Occupations (sorted by AI exposure)", ""]
    for score_val in range(10, -1, -1):
        group = [r for r in records if r["exposure"] == score_val]
        if not group:
            continue
        group_jobs = sum(r["jobs"] or 0 for r in group)
        lines.append(f"### Exposure {score_val}/10 ({len(group)} occupations, {fmt_jobs(group_jobs)} jobs)")
        lines.append("")
        lines.append("| Occupation | Pay | Jobs | Outlook | Education | Rationale |")
        lines.append("|-----------|-----|------|---------|-----------|-----------|")
        for r in group:
            outlook = f"{r['outlook_pct']:+d}%" if r["outlook_pct"] is not None else "?"
            edu_short = {
                "High school diploma or equivalent": "HS",
                "Bachelor's degree": "BS",
                "Master's degree": "MS",
                "Doctoral or professional degree": "PhD",
                "Associate's degree": "AS",
                "Postsecondary nondegree award": "Postsec",
                "No formal educational credential": "None",
            }.get(r["education"], r["education"][:8] if r["education"] else "?")
            rationale = r["rationale"].replace("|", "/").replace("\n", " ")[:100]
            lines.append(f"| {r['title']} | {fmt_pay(r['pay'])} | {fmt_jobs(r['jobs'])} | {outlook} | {edu_short} | {rationale} |")
        lines.append("")

    text = "\n".join(lines)
    with open("prompt.md", "w") as f:
        f.write(text)

    print(f"Wrote prompt.md ({len(text):,} chars, {len(lines):,} lines)")
    return text


prompt_text = generate_prompt_md()
# Preview first 2000 chars
print(prompt_text[:2000])

## 6. Quick Analysis & Charts

In [ ]:
# Scatter: Pay vs AI Exposure (bubble size = jobs)
import plotly.express as px

scatter_df = df.dropna(subset=["median_pay_annual", "exposure", "num_jobs_2024"]).copy()
scatter_df["size"] = (scatter_df["num_jobs_2024"] ** 0.4).clip(upper=50)  # sqrt scale for visibility
scatter_df["pay_k"] = scatter_df["median_pay_annual"] / 1000

fig_scatter = px.scatter(
    scatter_df,
    x="pay_k",
    y="exposure",
    size="size",
    color="category",
    hover_name="title",
    hover_data={"pay_k": ":.0f", "exposure": True, "num_jobs_2024": ":,", "size": False},
    labels={"pay_k": "Median Annual Pay ($K)", "exposure": "AI Exposure (0-10)"},
    title="Pay vs AI Exposure by Occupation<br><sup>Bubble size = number of jobs</sup>",
    height=600,
)
fig_scatter.update_layout(showlegend=True)
fig_scatter.show()

In [ ]:
# Top 20 largest occupations with their AI exposure
top20 = df.dropna(subset=["num_jobs_2024", "exposure"]).nlargest(20, "num_jobs_2024")

fig_bar = px.bar(
    top20.sort_values("num_jobs_2024"),
    x="num_jobs_2024",
    y="title",
    color="exposure",
    color_continuous_scale=[
        [0.0, "#1a9641"],
        [0.5, "#ffffbf"],
        [1.0, "#d7191c"],
    ],
    range_color=[0, 10],
    hover_data={"median_pay_annual": ":,.0f", "exposure": True},
    labels={"num_jobs_2024": "Number of Jobs (2024)", "title": "", "exposure": "AI Exposure"},
    title="Top 20 Largest US Occupations — AI Exposure<br><sup>Color = AI exposure score</sup>",
    height=650,
)
fig_bar.update_layout(yaxis=dict(tickfont=dict(size=11)))
fig_bar.show()

In [ ]:
# Declining occupations: BLS says shrinking AND high AI exposure
declining = df[
    (df["outlook_pct"].notna()) & (df["outlook_pct"] < 0) & (df["exposure"].notna())
].sort_values("outlook_pct")

print(f"Declining occupations: {len(declining)}")
print(f"\n{'Occupation':<45} {'Outlook':>8} {'AI Exp':>7} {'Jobs':>12}")
print("-" * 76)
for _, r in declining.iterrows():
    jobs_str = f"{r['num_jobs_2024']:,.0f}" if pd.notna(r['num_jobs_2024']) else "?"
    print(f"{r['title']:<45} {r['outlook_pct']:>+7.0f}% {r['exposure']:>6.0f}/10 {jobs_str:>12}")

In [ ]:
# Avg AI exposure by pay band (job-weighted)
import numpy as np

pay_bands = [
    ("<$35K",      0,      35000),
    ("$35-50K",    35000,  50000),
    ("$50-75K",    50000,  75000),
    ("$75-100K",   75000,  100000),
    ("$100-150K",  100000, 150000),
    ("$150K+",     150000, float("inf")),
]

print(f"{'Pay band':<15} {'Avg AI Exposure':>16} {'Occupations':>12} {'Total Jobs':>12}")
print("-" * 58)
for name, lo, hi in pay_bands:
    mask = (
        df["median_pay_annual"].notna() &
        (df["median_pay_annual"] >= lo) &
        (df["median_pay_annual"] < hi) &
        df["exposure"].notna() &
        df["num_jobs_2024"].notna()
    )
    g = df[mask]
    if len(g) == 0:
        continue
    w_avg = (g["exposure"] * g["num_jobs_2024"]).sum() / g["num_jobs_2024"].sum()
    total_j = g["num_jobs_2024"].sum()
    print(f"{name:<15} {w_avg:>14.1f}/10 {len(g):>12} {total_j:>12,.0f}")